# Replicación de *Next Day Wildfire Spread* (Huot et al., 2022)

**Artículo:** Huot, F., Hu, R. L., Goyal, N., Sankar, T., Ihme, M., & Chen, Y.-F. (2022). Next Day Wildfire Spread: A Machine Learning Dataset to Predict Wildfire Spreading From Remote-Sensing Data. *IEEE Transactions on Geoscience and Remote Sensing, 60*, Art. 4412513. https://doi.org/10.1109/TGRS.2022.3192974

**Qué hace este notebook:** repite el flujo de trabajo del artículo con el dataset que publicaron los autores en Kaggle y compara los resultados con los que reportan.

| Paso | Qué se hace | Sección del artículo |
| --- | --- | --- |
| 0 | Preparar el entorno y definir el código | — |
| 1 | Explorar el dataset | II |
| 2 | Entrenar y evaluar la red neuronal | III, IV y V-A |
| 3 | Modelos base: persistencia, regresión logística, random forest | IV-A, Apéndice A |
| 4 | Ablación de la red (¿qué variables necesita?) | V-B |
| 5 | Comparar con el artículo | Tablas II a VIII |

**Dónde correrlo**

- **Kaggle (recomendado):** *Add input* → buscar `next-day-wildfire-spread` (fantineh) → activar *Accelerator: GPU*.
- **Google Colab:** *Entorno de ejecución → Cambiar tipo → GPU*. El dataset se descarga solo (2 GB).

Con GPU (Kaggle, NVIDIA T4) cada una de las tres redes se entrena en unos 12 minutos. Sin GPU, el notebook completo tarda entre 3 y 4 horas (cerca de 1 hora por red en CPU).


## Paso 0 — Preparar el entorno y definir el código

In [ ]:
# El dataset de Kaggle es público: kagglehub lo descarga sin usuario ni clave.
# En Kaggle no hace falta descargar nada si se agregó el dataset como "input".
%pip install -q kagglehub

import os

# Presupuesto de cómputo: "prueba" (segundos), "prototipo" (el de la
# replicación) o "largo" (más épocas, para acercarse al artículo).
os.environ.setdefault("WILDFIRE_PRESUPUESTO", "prototipo")

# En un notebook no existe __file__; se define para que el código que arma las
# rutas del proyecto funcione igual que en los archivos .py.
os.makedirs("src", exist_ok=True)
__file__ = os.path.abspath("src/notebook.py")

import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "no hay (se usará CPU)")


### Configuración (`src/config.py`)
Nombres de variables, estadísticas, hiperparámetros, presupuesto de cómputo y los resultados del artículo.

In [ ]:
"""
config.py — Configuración central del prototipo de replicación.

Artículo replicado:
    Huot, F., Hu, R. L., Goyal, N., Sankar, T., Ihme, M., & Chen, Y.-F. (2022).
    Next Day Wildfire Spread: A Machine Learning Dataset to Predict Wildfire
    Spreading From Remote-Sensing Data. IEEE TGRS, 60, Art. 4412513.
    https://doi.org/10.1109/TGRS.2022.3192974

Aquí vive todo lo que NO es lógica: nombres de variables, estadísticas de
normalización, hiperparámetros, el presupuesto de cómputo y los resultados que
reporta el artículo (para compararlos con los nuestros al final).

Cada constante indica de dónde sale:
    [ART]  = texto del artículo (sección o tabla)
    [COD]  = código publicado por los autores en GitHub (constants.py, etc.)
    [REP]  = decisión propia de esta replicación (el artículo no lo especifica)
"""

import os

# ---------------------------------------------------------------------------
# 1. RUTAS
# ---------------------------------------------------------------------------
# Carpeta raíz del proyecto (la que contiene a src/).
RAIZ = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

# Identificador del dataset en Kaggle, publicado por los mismos autores.
DATASET_KAGGLE = "fantineh/next-day-wildfire-spread"

# Presupuesto de cómputo elegido (se explica en la sección 5 de este archivo).
PRESUPUESTO = os.environ.get("WILDFIRE_PRESUPUESTO", "prototipo")

# Carpeta donde se guardan tablas (JSON/CSV), figuras (PNG) y el modelo. Las
# corridas de "prueba" van a otra carpeta para no pisar los resultados reales.
DIR_RESULTADOS = os.path.join(
    RAIZ, "resultados" if PRESUPUESTO != "prueba" else "resultados_prueba"
)
DIR_MODELOS = os.path.join(DIR_RESULTADOS, "modelos")

# ---------------------------------------------------------------------------
# 2. VARIABLES DEL DATASET  [COD: constants.py]
# ---------------------------------------------------------------------------
# Las 12 capas de entrada, en el MISMO orden que usa el código de los autores.
# Cada una es una imagen de 64 x 64 píxeles (1 píxel = 1 km).
VARIABLES_ENTRADA = (
    "elevation",     # elevación del terreno (m)                    — SRTM
    "pdsi",          # índice de sequía de Palmer                   — GRIDMET Drought
    "NDVI",          # índice de vegetación (x 10 000)              — VIIRS
    "pr",            # precipitación diaria (mm)                    — GRIDMET
    "sph",           # humedad específica (kg/kg)                   — GRIDMET
    "th",            # dirección del viento (grados desde el norte) — GRIDMET
    "tmmn",          # temperatura mínima (K)                       — GRIDMET
    "tmmx",          # temperatura máxima (K)                       — GRIDMET
    "vs",            # velocidad del viento (m/s)                   — GRIDMET
    "erc",           # energy release component (índice NFDRS)      — GRIDMET
    "population",    # densidad de población (hab/km²)              — GPWv4
    "PrevFireMask",  # máscara de fuego del día t ("fuego de hoy")  — MODIS
)

# La etiqueta: máscara de fuego del día t + 1 ("fuego de mañana").
VARIABLE_SALIDA = "FireMask"

# Nombres en español para tablas y figuras.
NOMBRES_ES = {
    "elevation": "Elevación",
    "pdsi": "Sequía (PDSI)",
    "NDVI": "Vegetación (NDVI)",
    "pr": "Precipitación",
    "sph": "Humedad",
    "th": "Dirección del viento",
    "tmmn": "Temp. mínima",
    "tmmx": "Temp. máxima",
    "vs": "Velocidad del viento",
    "erc": "ERC",
    "population": "Población",
    "PrevFireMask": "Fuego previo (día t)",
    "FireMask": "Fuego (día t+1)",
}

# Nombre que usa el artículo para cada variable (Tablas IV a VIII).
NOMBRES_ARTICULO = {
    "elevation": "Elevation",
    "pdsi": "Drought",
    "NDVI": "Vegetation",
    "pr": "Precipitation",
    "sph": "Humidity",
    "th": "Wind direction",
    "tmmn": "Min temperature",
    "tmmx": "Max temperature",
    "vs": "Wind speed",
    "erc": "ERC",
    "population": "Population",
    "PrevFireMask": "Previous fire mask",
}

# Estadísticas para limpiar y normalizar cada variable  [COD: constants.py].
# Formato: (recorte_mínimo, recorte_máximo, media, desviación_estándar).
# Los autores las calcularon SOLO con el conjunto de entrenamiento, después de
# recortar (Sección III del artículo). Los recortes son el percentil 0,1 y 99,9
# o un límite físico (p. ej. la precipitación no puede ser negativa).
ESTADISTICAS = {
    "elevation": (0.0, 3141.0, 657.0, 649.0),
    "pdsi": (-6.1, 7.9, 0.0, 2.7),
    "NDVI": (-9821.0, 9996.0, 5158.0, 2467.0),
    "pr": (0.0, 44.5, 1.7, 4.5),
    "sph": (0.0, 1.0, 0.0072, 0.0043),
    "th": (0.0, 360.0, 190.3, 72.6),
    "tmmn": (253.2, 298.9, 281.1, 9.0),
    "tmmx": (253.2, 315.1, 295.2, 9.8),
    "vs": (0.0, 10.0, 3.9, 1.4),
    "erc": (0.0, 106.0, 37.0, 21.0),
    "population": (0.0, 2534.0, 26.0, 155.0),
    # Las máscaras de fuego NO se normalizan: -1 incierto, 0 sin fuego, 1 fuego.
    "PrevFireMask": (-1.0, 1.0, 0.0, 1.0),
    "FireMask": (-1.0, 1.0, 0.0, 1.0),
}

# ---------------------------------------------------------------------------
# 3. TAMAÑOS  [ART: Secciones II y III]
# ---------------------------------------------------------------------------
TAM_ORIGINAL = 64   # cada muestra del dataset mide 64 x 64 km
TAM_RECORTE = 32    # la red se entrena con recortes aleatorios de 32 x 32 km

# ---------------------------------------------------------------------------
# 4. HIPERPARÁMETROS DE LA RED NEURONAL  [ART: Apéndice A-A]
# ---------------------------------------------------------------------------
FILTROS_CODIFICADOR = (16, 32)   # 16 filtros al inicio, 32 en el centro
FILTROS_DECODIFICADOR = (32, 16)  # simétrico al codificador
TASA_DROPOUT = 0.1               # "the dropout rate is 0.1"
TASA_APRENDIZAJE = 1e-4          # "learning rate of 0.0001"
PESO_CLASE_FUEGO = 3.0           # "a weight of 3 in the fire class"
TAM_LOTE = 32                    # [REP] el artículo exploró 32, 64, 128 y 256

# ---------------------------------------------------------------------------
# 5. PRESUPUESTO DE CÓMPUTO  [REP]
# ---------------------------------------------------------------------------
# El artículo entrenó 1000 épocas x 1000 pasos = 1 000 000 de pasos en 4 GPU
# V100. Eso es imposible en un prototipo de clase, así que se reduce. La
# variable de entorno WILDFIRE_PRESUPUESTO permite cambiarlo sin tocar el código:
#     "prueba"    -> segundos, solo para verificar que todo corre
#     "prototipo" -> el que se usó para los resultados de esta replicación
#     "largo"     -> para quien tenga GPU y quiera acercarse más al artículo
# Una época = 468 pasos de 32 muestras = una pasada por las 14 979 muestras de
# entrenamiento. "prototipo" son 28 080 pasos: el 2,8 % de los del artículo.
PRESUPUESTOS = {
    "prueba":    dict(epocas=2,   pasos_por_epoca=20,  pixeles_logistica=40_000,    pixeles_bosque=20_000),
    "prototipo": dict(epocas=60,  pasos_por_epoca=468, pixeles_logistica=1_000_000, pixeles_bosque=1_000_000),
    "largo":     dict(epocas=300, pasos_por_epoca=468, pixeles_logistica=1_000_000, pixeles_bosque=1_000_000),
}

# Semilla para que los experimentos se puedan repetir.
SEMILLA = 123

# [REP] Recortes de 32 x 32 km que se sacan de cada muestra de prueba. El
# artículo usa uno; con tres la medición depende menos del azar del recorte.
REPETICIONES_PRUEBA = 3

# ---------------------------------------------------------------------------
# 6. HIPERPARÁMETROS DE LOS MODELOS BASE  [ART: Apéndice A-B]
# ---------------------------------------------------------------------------
PROFUNDIDAD_BOSQUE = 15      # "we select a maximum depth of 15"
W_LOGISTICA = 7              # factor W del peso de la clase "sin fuego"
W_BOSQUE = 5
# [REP] Tope de píxeles de entrenamiento para los modelos base (mitad fuego,
# mitad sin fuego). El artículo no dice cuántos usó; estos topes mantienen el
# tiempo de cómputo razonable en CPU y dependen del presupuesto elegido.
MAX_PIXELES_LOGISTICA = PRESUPUESTOS[PRESUPUESTO]["pixeles_logistica"]
MAX_PIXELES_BOSQUE = PRESUPUESTOS[PRESUPUESTO]["pixeles_bosque"]

# ---------------------------------------------------------------------------
# 7. RESULTADOS DEL ARTÍCULO (para comparar)  [ART: Tablas II a VIII]
# ---------------------------------------------------------------------------
# Todos en porcentaje: (AUC PR, precisión, recall).
ARTICULO_TABLA_II = {
    "Red neuronal": (28.4, 33.6, 43.1),
    "Random forest": (22.5, 26.3, 46.9),
    "Regresión logística": (19.8, 32.5, 35.3),
    "Persistencia": (11.5, 35.7, 27.3),
}

# Tabla III, filas de "Persistence": resolución más gruesa (etiqueta binaria).
ARTICULO_TABLA_III_PERSISTENCIA = {
    2: (19.5, 45.6, 36.8),
    4: (30.6, 56.0, 46.8),
    8: (41.3, 64.0, 52.5),
}
# Tabla III, filas "Binary" de la red neuronal (referencia, no se replican).
ARTICULO_TABLA_III_RED_BINARIA = {
    2: (39.8, 40.3, 51.9),
    4: (53.5, 48.8, 61.0),
    8: (66.3, 62.1, 63.1),
}

# Tabla IV: red neuronal re-entrenada QUITANDO una variable.
ARTICULO_TABLA_IV = {
    "PrevFireMask": (6.8, 4.6, 0.2),
    "sph": (26.1, 35.4, 35.7),
    "tmmx": (26.8, 35.4, 36.6),
    "vs": (27.2, 33.1, 42.4),
    "NDVI": (27.7, 35.0, 40.0),
    "pr": (28.0, 32.5, 42.0),
    "population": (28.1, 32.8, 43.0),
    "th": (28.3, 32.8, 43.1),
    "tmmn": (28.6, 35.2, 40.9),
    "elevation": (28.8, 33.6, 43.0),
    "pdsi": (28.8, 36.8, 39.0),
    "erc": (28.8, 33.0, 43.2),
}

# Tabla V: red neuronal con el fuego previo + UNA sola variable más.
ARTICULO_TABLA_V = {
    "NDVI": (28.2, 33.0, 41.4),
    "elevation": (27.0, 31.3, 44.4),
    "tmmx": (26.7, 32.7, 43.5),
    "population": (26.6, 32.6, 43.2),
    "pr": (26.6, 34.1, 40.6),
    "tmmn": (26.2, 29.9, 47.2),
    "pdsi": (26.0, 30.8, 44.1),
    "vs": (25.9, 30.3, 45.6),
    "erc": (25.8, 30.7, 45.0),
    "sph": (25.7, 32.4, 41.7),
    "th": (25.5, 30.9, 43.5),
}

# Sección V-B: red neuronal usando SOLO el fuego previo.
ARTICULO_SOLO_FUEGO_PREVIO = (22.9, 29.2, 49.6)

# Tabla VI: las 20 variables más importantes del random forest (Gini).
# El número entre paréntesis es la posición del píxel en la vecindad de 3 x 3
# (1 = arriba-izquierda, 5 = centro, 9 = abajo-derecha).
ARTICULO_TABLA_VI = [
    ("PrevFireMask", 5, 0.092), ("PrevFireMask", 6, 0.074),
    ("PrevFireMask", 4, 0.073), ("PrevFireMask", 2, 0.071),
    ("PrevFireMask", 1, 0.068), ("PrevFireMask", 8, 0.067),
    ("PrevFireMask", 9, 0.061), ("PrevFireMask", 3, 0.052),
    ("PrevFireMask", 7, 0.051), ("erc", 7, 0.017), ("erc", 8, 0.014),
    ("erc", 5, 0.014), ("erc", 6, 0.012), ("erc", 9, 0.010),
    ("erc", 2, 0.008), ("erc", 1, 0.008), ("erc", 4, 0.008),
    ("erc", 3, 0.007), ("elevation", 7, 0.007), ("elevation", 9, 0.005),
]

# Tabla VIII: regresión logística re-entrenada QUITANDO una variable.
ARTICULO_TABLA_VIII = {
    "PrevFireMask": (3.1, 0.0, 0.0),
    "erc": (19.6, 32.4, 35.5),
    "elevation": (19.6, 32.3, 35.5),
    "pdsi": (19.6, 32.4, 35.4),
    "tmmn": (19.6, 32.2, 35.5),
    "th": (19.7, 32.3, 35.7),
    "tmmx": (19.7, 32.3, 35.5),
    "NDVI": (19.8, 32.3, 35.6),
    "pr": (19.8, 32.3, 35.6),
    "population": (19.8, 32.3, 35.6),
    "sph": (19.8, 32.4, 35.6),
    "vs": (20.0, 32.1, 35.7),
}

# Datos generales del dataset según el artículo (Sección II-C).
ARTICULO_DATASET = {
    "muestras": 18545,
    "fuego_crece": 10798,     # 58 %
    "fuego_decrece": 7191,    # 39 %
}


### Datos (`src/datos.py`)
Lectura de los TFRecord, aumento de datos, recorte y normalización.

In [ ]:
"""
datos.py — Lectura y preparación del dataset "Next Day Wildfire Spread".

Reproduce la Sección III del artículo ("Data Preprocessing") y el archivo
dataset.py del código de los autores:

    1. Leer los archivos TFRecord publicados en Kaggle.
    2. Aumentar los datos (solo al entrenar): volteo, rotación y recorte
       aleatorio de 32 x 32 km.
    3. Recortar valores extremos y normalizar cada variable.

Cada muestra del dataset es un cuadrado de 64 x 64 km con 12 capas de entrada
y 1 capa de salida (la máscara de fuego del día siguiente).
"""

import glob
import os

import numpy as np
import tensorflow as tf


# ---------------------------------------------------------------------------
# 1. LOCALIZAR Y LEER LOS ARCHIVOS
# ---------------------------------------------------------------------------


def localizar_dataset():
    """Devuelve la carpeta que contiene los .tfrecord del dataset.

    Busca, en este orden:
      1. La variable de entorno WILDFIRE_DATOS (si el usuario ya los tiene).
      2. /kaggle/input/... (cuando se corre dentro de un notebook de Kaggle).
      3. La carpeta local datos/ del proyecto.
      4. Si no están, los descarga de Kaggle con kagglehub (dataset público,
         no necesita usuario ni contraseña).
    """
    candidatas = [
        os.environ.get("WILDFIRE_DATOS", ""),
        "/kaggle/input/next-day-wildfire-spread",
        os.path.join(RAIZ, "datos"),
    ]
    for carpeta in candidatas:
        if carpeta and glob.glob(os.path.join(carpeta, "*train*.tfrecord")):
            return carpeta

    import kagglehub  # se importa aquí porque solo hace falta para descargar

    return kagglehub.dataset_download(DATASET_KAGGLE)


def archivos(particion):
    """Lista ordenada de TFRecords de una partición: 'train', 'eval' o 'test'.

    La separación 80/10/10 ya viene hecha por los autores (por semanas al azar
    entre 2012 y 2020), así que se respeta tal cual para poder comparar.
    """
    patron = os.path.join(localizar_dataset(), f"*{particion}*.tfrecord")
    lista = sorted(glob.glob(patron))
    if not lista:
        raise FileNotFoundError(f"No se encontraron archivos con el patrón {patron}")
    return lista


def _parsear(ejemplo_serializado):
    """Convierte un registro binario en dos tensores SIN normalizar.

    Returns:
        x: (64, 64, 12) con las variables de entrada en sus unidades originales.
        y: (64, 64, 1) con la máscara de fuego de mañana: 1 fuego, 0 sin fuego,
           -1 incierto (nubes o dato sin procesar).
    """
    nombres = list(VARIABLES_ENTRADA) + [VARIABLE_SALIDA]
    descripcion = {
        n: tf.io.FixedLenFeature([TAM_ORIGINAL, TAM_ORIGINAL], tf.float32)
        for n in nombres
    }
    campos = tf.io.parse_single_example(ejemplo_serializado, descripcion)
    x = tf.stack([campos[n] for n in VARIABLES_ENTRADA], axis=-1)
    y = campos[VARIABLE_SALIDA][..., tf.newaxis]
    return x, y


def leer_crudo(particion):
    """tf.data.Dataset con las muestras de una partición, sin normalizar."""
    return tf.data.TFRecordDataset(archivos(particion)).map(
        _parsear, num_parallel_calls=tf.data.AUTOTUNE
    )


# ---------------------------------------------------------------------------
# 2. LIMPIEZA Y NORMALIZACIÓN  (Sección III del artículo)
# ---------------------------------------------------------------------------


def _vectores_estadisticos(variables):
    """Arma 4 vectores (mín, máx, media, desv.) alineados con `variables`."""
    tabla = np.array([ESTADISTICAS[v] for v in variables], dtype=np.float32)
    return tabla[:, 0], tabla[:, 1], tabla[:, 2], tabla[:, 3]


def normalizar(x, variables=VARIABLES_ENTRADA):
    """Recorta valores extremos y estandariza cada canal: (x - media) / desv.

    El recorte evita valores físicamente imposibles y rangos enormes que
    desestabilizan el entrenamiento. La máscara de fuego previo no cambia
    porque sus "estadísticas" son (-1, 1, 0, 1). Funciona igual con tensores de
    TensorFlow que con arreglos de NumPy.
    """
    minimo, maximo, media, desv = _vectores_estadisticos(variables)
    x = tf.clip_by_value(x, minimo, maximo)
    return (x - media) / desv


def seleccionar_variables(x, variables):
    """Se queda solo con los canales pedidos (para los estudios de ablación)."""
    indices = [VARIABLES_ENTRADA.index(v) for v in variables]
    return tf.gather(x, indices, axis=-1)


# ---------------------------------------------------------------------------
# 3. AUMENTO DE DATOS  (Secciones III y IV-B del artículo)
# ---------------------------------------------------------------------------

_INDICE_VIENTO = VARIABLES_ENTRADA.index("th")


def _cambiar_direccion_viento(x, funcion):
    """Aplica `funcion` al canal de dirección del viento (en grados)."""
    canales = tf.unstack(x, axis=-1)
    canales[_INDICE_VIENTO] = funcion(canales[_INDICE_VIENTO])
    return tf.stack(canales, axis=-1)


def aumentar(x, y):
    """Volteo y rotación al azar de una muestra SIN normalizar.

    Voltear o girar la imagen también cambia hacia dónde "apunta" el viento,
    así que el canal de dirección (grados en sentido horario desde el norte)
    se corrige con las mismas fórmulas del código de los autores
    (image_utils.py). Por eso se aumenta ANTES de normalizar: las fórmulas
    necesitan los grados reales.

    Entrada y etiqueta se transforman juntas para que sigan alineadas.
    """
    # Volteo izquierda-derecha (50 %): el este pasa a ser oeste.
    if tf.random.uniform(()) < 0.5:
        x = tf.image.flip_left_right(x)
        y = tf.image.flip_left_right(y)
        x = _cambiar_direccion_viento(x, lambda g: (360.0 - g) % 360.0)

    # Volteo arriba-abajo (50 %): el norte pasa a ser sur.
    if tf.random.uniform(()) < 0.5:
        x = tf.image.flip_up_down(x)
        y = tf.image.flip_up_down(y)
        x = _cambiar_direccion_viento(x, lambda g: (180.0 - g) % 360.0)

    # Rotación de 0, 90, 180 o 270 grados en sentido antihorario.
    k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
    x = tf.image.rot90(x, k)
    y = tf.image.rot90(y, k)
    giro = 90.0 * tf.cast(k, tf.float32)
    x = _cambiar_direccion_viento(x, lambda g: (g - giro) % 360.0)
    return x, y


def recorte_aleatorio(x, y, tam=TAM_RECORTE):
    """Recorta la misma ventana de tam x tam en la entrada y en la etiqueta.

    En el dataset el fuego casi siempre está en el centro del cuadro de
    64 x 64 km. El recorte aleatorio lo mueve de sitio, para que la red no
    aprenda simplemente "el fuego está en el medio" (Sección III).
    """
    canales_x = x.shape[-1]
    junto = tf.concat([x, y], axis=-1)
    junto = tf.image.random_crop(junto, [tam, tam, junto.shape[-1]])
    return junto[..., :canales_x], junto[..., canales_x:]


# ---------------------------------------------------------------------------
# 4. DATASETS LISTOS PARA USAR
# ---------------------------------------------------------------------------

def dataset_entrenamiento(variables=VARIABLES_ENTRADA, lote=TAM_LOTE, semilla=SEMILLA,
                          tam_barajado=2000):
    """Flujo infinito de lotes para entrenar la red neuronal.

    barajar archivos -> leer 4 archivos a la vez -> barajar muestras
        -> aumentar -> recortar -> normalizar -> elegir variables -> lotes

    Los datos se leen del disco en cada pasada (no se guardan en memoria), así
    el entrenamiento ocupa poca RAM. El orden se mezcla dos veces: primero el
    de los 15 archivos y luego el de las muestras, con una ventana de
    `tam_barajado` muestras (igual que el `shuffle_buffer_size` de los autores).
    """

    def preparar(x, y):
        x, y = aumentar(x, y)
        x, y = recorte_aleatorio(x, y)
        x = normalizar(x)
        x = seleccionar_variables(x, variables)
        return x, y

    rutas = tf.data.Dataset.from_tensor_slices(archivos("train"))
    rutas = rutas.shuffle(len(archivos("train")), seed=semilla).repeat()
    ds = rutas.interleave(
        tf.data.TFRecordDataset, cycle_length=4, num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.map(_parsear, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.shuffle(tam_barajado, seed=semilla)
    ds = ds.map(preparar, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(lote).prefetch(tf.data.AUTOTUNE)


def cargar_numpy(particion):
    """Carga una partición completa en memoria, ya normalizada.

    Se usa para validación y prueba (son pequeñas: ~1 900 y ~1 700 muestras).

    Returns:
        X: (N, 64, 64, 12) float32 normalizado.
        Y: (N, 64, 64, 1) float32 con valores -1, 0 o 1.
    """
    xs, ys = [], []
    for x, y in leer_crudo(particion).batch(512):
        xs.append(normalizar(x).numpy())
        ys.append(y.numpy())
    return np.concatenate(xs), np.concatenate(ys)


def iterar_numpy(particion, tam_bloque=1024):
    """Recorre una partición por bloques normalizados (sin cargarla entera).

    Lo usan los modelos base para muestrear píxeles del conjunto de
    entrenamiento sin ocupar los ~3 GB que pesa completo.
    """
    for x, y in leer_crudo(particion).batch(tam_bloque):
        yield normalizar(x).numpy(), y.numpy()


def recortes_fijos(X, Y, tam=TAM_RECORTE, repeticiones=1, semilla=SEMILLA):
    """Recortes aleatorios pero REPETIBLES (misma semilla = mismos recortes).

    Los autores evalúan sobre recortes aleatorios de 32 x 32 km (dataset.py,
    modo PREDICT con random_crop). Fijar la semilla permite que todos los
    modelos se midan exactamente sobre los mismos píxeles.

    Args:
        repeticiones: cuántos recortes distintos sacar de cada muestra.

    Returns:
        Xr: (N * repeticiones, tam, tam, C)
        Yr: (N * repeticiones, tam, tam, 1)
    """
    azar = np.random.default_rng(semilla)
    n, alto, ancho, _ = X.shape
    xs, ys = [], []
    for _ in range(repeticiones):
        filas = azar.integers(0, alto - tam + 1, size=n)
        columnas = azar.integers(0, ancho - tam + 1, size=n)
        xs.append(np.stack([X[i, f:f + tam, c:c + tam] for i, (f, c) in enumerate(zip(filas, columnas))]))
        ys.append(np.stack([Y[i, f:f + tam, c:c + tam] for i, (f, c) in enumerate(zip(filas, columnas))]))
    return np.concatenate(xs), np.concatenate(ys)


### Modelo (`src/modelo.py`)
Autoencoder convolucional de la Fig. 6 del artículo.

In [ ]:
"""
modelo.py — Autoencoder convolucional para segmentar "fuego de mañana".

Reproduce la Fig. 6 y el Apéndice A-A del artículo. La estructura de capas se
tomó del código de los autores (models/cnn_autoencoder_model.py y
models/model_utils.py) y se reescribió para Keras 3.

Idea general (Fig. 2 del artículo):

    entrada 32x32x12 ──► CODIFICADOR ──► cuello de botella 8x8x32 ──► DECODIFICADOR ──► salida 32x32x1
                         (comprime)                                   (reconstruye)

La entrada es una "foto" de 12 canales (11 variables + el fuego de hoy) y la
salida es un mapa con un número por píxel (logit). Al pasarlo por una sigmoide
se convierte en la probabilidad de que ese km² tenga fuego mañana.
"""

import keras
from keras import layers


# Pendiente de la LeakyReLU. 0.3 era el valor por defecto de Keras cuando se
# escribió el código original; se fija aquí para que no dependa de la versión.
PENDIENTE_LEAKY = 0.3


def conv(filtros, tam_nucleo=3):
    """Convolución 2D como la definen los autores.

    Núcleo de 3x3, paso 1, relleno 'same' (no cambia el tamaño de la imagen) y
    SIN término de sesgo (use_bias=False en model_utils.py).
    """
    return layers.Conv2D(
        filtros, tam_nucleo, strides=1, padding="same", use_bias=False
    )


def bloque_inicial(x, filtros, dropout):
    """Primer bloque del codificador (parte superior izquierda de la Fig. 6).

    Camino principal:  Conv -> LeakyReLU -> Dropout -> Conv
    Atajo (shortcut):  Conv 1x1 -> Dropout
    Al final se suman. El atajo deja pasar la entrada casi intacta, lo que
    facilita el entrenamiento (conexión residual).
    """
    principal = conv(filtros)(x)
    principal = layers.LeakyReLU(negative_slope=PENDIENTE_LEAKY)(principal)
    principal = layers.Dropout(dropout)(principal)
    principal = conv(filtros)(principal)

    atajo = conv(filtros, tam_nucleo=1)(x)
    atajo = layers.Dropout(dropout)(atajo)
    return layers.Add()([atajo, principal])


def bloque_residual(x, filtros, dropout, reducir):
    """ResBlock de la Fig. 6 (derecha).

    reducir=True  -> variante "Maxpool": baja la resolución a la mitad.
                     Se usa en el codificador y en el cuello de botella.
    reducir=False -> variante "Conv2D": mantiene la resolución.
                     Se usa en el decodificador.

    Camino principal:
        LeakyReLU -> Dropout -> (MaxPool 2x2  o  Conv 3x3)
        LeakyReLU -> Dropout -> Conv 3x3 -> Dropout
    Atajo:
        Conv 1x1 (con paso 2 si se reduce, para que los tamaños coincidan)
    """
    principal = layers.LeakyReLU(negative_slope=PENDIENTE_LEAKY)(x)
    principal = layers.Dropout(dropout)(principal)
    if reducir:
        principal = layers.MaxPooling2D(pool_size=2, strides=2)(principal)
    else:
        principal = conv(filtros)(principal)

    principal = layers.LeakyReLU(negative_slope=PENDIENTE_LEAKY)(principal)
    principal = layers.Dropout(dropout)(principal)
    principal = conv(filtros)(principal)
    principal = layers.Dropout(dropout)(principal)

    atajo = layers.Conv2D(
        filtros, 1, strides=2 if reducir else 1, padding="same", use_bias=False
    )(x)
    return layers.Add()([atajo, principal])


def crear_modelo(
    num_canales,
    filtros_codificador=FILTROS_CODIFICADOR,
    filtros_decodificador=FILTROS_DECODIFICADOR,
    dropout=TASA_DROPOUT,
):
    """Construye el autoencoder convolucional del artículo.

    Args:
        num_canales: número de variables de entrada (12 con todas; menos en
            los experimentos de ablación).
        filtros_codificador: filtros del bloque inicial y de cada ResBlock del
            codificador. (16, 32) = configuración del artículo.
        filtros_decodificador: filtros de cada ResBlock del decodificador.
        dropout: fracción de neuronas que se apagan al entrenar.

    Returns:
        keras.Model que recibe (alto, ancho, num_canales) y devuelve
        (alto, ancho, 1) con LOGITS (sin sigmoide).

    El alto y el ancho se dejan libres (None): como la red solo tiene
    convoluciones, se puede entrenar con recortes de 32x32 y luego evaluar con
    la imagen completa de 64x64. Solo deben ser múltiplos de 4.
    """
    entrada = keras.Input(shape=(None, None, num_canales), name="variables")

    # ---- CODIFICADOR: 32x32 -> 16x16 ------------------------------------
    x = bloque_inicial(entrada, filtros_codificador[0], dropout)
    for filtros in filtros_codificador[1:]:
        x = bloque_residual(x, filtros, dropout, reducir=True)

    # ---- CUELLO DE BOTELLA: 16x16 -> 8x8 --------------------------------
    # Segundo "ResBlock (Maxpool)" de la Fig. 6.
    x = bloque_residual(x, filtros_codificador[-1], dropout, reducir=True)

    # ---- DECODIFICADOR: 8x8 -> 16x16 -> 32x32 ---------------------------
    # Cada etapa agranda la imagen al doble (vecino más cercano) y la refina
    # con un "ResBlock (Conv2D)".
    for filtros in filtros_decodificador:
        x = layers.UpSampling2D(size=2, interpolation="nearest")(x)
        x = bloque_residual(x, filtros, dropout, reducir=False)

    # ---- CAPA DE SALIDA: 1 filtro = 1 logit por píxel -------------------
    salida = conv(1, tam_nucleo=1)(x)
    return keras.Model(entrada, salida, name="autoencoder_convolucional")


### Pérdida y métricas (`src/metricas.py`)
Entropía cruzada ponderada y AUC PR / precisión / recall, ignorando píxeles inciertos.

In [ ]:
"""
metricas.py — Función de pérdida y métricas, ignorando los píxeles inciertos.

Reproduce models/losses.py y models/metrics.py del código de los autores y la
Sección IV-B del artículo.

Dos ideas clave:

1. DESBALANCE. Solo ~1 % de los píxeles tiene fuego. Una red que siempre diga
   "no hay fuego" acertaría el 99 % de las veces y no serviría para nada. Por
   eso (a) la pérdida castiga más los errores en píxeles con fuego (peso 3) y
   (b) no se usa accuracy sino AUC PR, precisión y recall.

2. ETIQUETAS INCIERTAS. Los píxeles con valor -1 (nubes, dato faltante) no
   cuentan ni para la pérdida ni para las métricas.
"""

import keras
import numpy as np
import tensorflow as tf


def perdida_ponderada_con_mascara(peso_fuego=PESO_CLASE_FUEGO):
    """Entropía cruzada binaria con peso para la clase fuego y máscara.

    Para cada píxel con etiqueta z (0 o 1) y logit x:
        pérdida = -[ peso * z * log(sigmoide(x)) + (1 - z) * log(1 - sigmoide(x)) ]
    Los píxeles inciertos (z = -1) se multiplican por 0.
    """

    def perdida(y_verdad, logits):
        mascara = tf.cast(tf.not_equal(y_verdad, -1.0), tf.float32)
        etiquetas = tf.maximum(y_verdad, 0.0)  # el -1 pasa a 0; igual se anula
        por_pixel = tf.nn.weighted_cross_entropy_with_logits(
            labels=etiquetas, logits=logits, pos_weight=peso_fuego
        )
        return tf.reduce_mean(mascara * por_pixel)

    return perdida


class _ConMascara:
    """Mezcla que adapta una métrica de Keras: aplica sigmoide y máscara."""

    def update_state(self, y_verdad, logits, sample_weight=None):
        probabilidad = tf.sigmoid(logits)
        mascara = tf.cast(tf.not_equal(y_verdad, -1.0), tf.float32)
        super().update_state(
            tf.maximum(y_verdad, 0.0), probabilidad, sample_weight=mascara
        )


class AUCConMascara(_ConMascara, keras.metrics.AUC):
    """AUC que ignora los píxeles inciertos (se usa con curve='PR')."""


class PrecisionConMascara(_ConMascara, keras.metrics.Precision):
    """De los píxeles que el modelo marcó como fuego, ¿cuántos lo eran?"""


class RecallConMascara(_ConMascara, keras.metrics.Recall):
    """De los píxeles que sí tenían fuego, ¿cuántos encontró el modelo?"""


def metricas_de_entrenamiento():
    """Métricas que Keras calcula en cada época (los nombres van al historial)."""
    return [
        AUCConMascara(curve="PR", name="auc_pr"),
        PrecisionConMascara(name="precision"),
        RecallConMascara(name="recall"),
    ]


def calcular_metricas(y_verdad, probabilidad, umbral=0.5):
    """Calcula AUC PR, precisión y recall (en %) para CUALQUIER modelo.

    Se usa la misma métrica de Keras que los autores (tf.keras.metrics.AUC con
    curve='PR' y 200 umbrales), de modo que la red, la regresión logística, el
    random forest y la persistencia se midan con la misma regla.

    Args:
        y_verdad: arreglo con valores -1 (incierto), 0 (sin fuego) o 1 (fuego).
        probabilidad: arreglo de la misma forma con valores entre 0 y 1.
        umbral: desde qué probabilidad se declara "fuego" (0.5, como Keras).

    Returns:
        dict con auc_pr, precision y recall en porcentaje, y los conteos.
    """
    y = np.asarray(y_verdad).reshape(-1)
    p = np.asarray(probabilidad, dtype=np.float32).reshape(-1)
    validos = y != -1  # se descartan los píxeles inciertos
    y, p = y[validos].astype(np.float32), np.clip(p[validos], 0.0, 1.0)

    auc = keras.metrics.AUC(curve="PR")
    for inicio in range(0, len(y), 2_000_000):  # por bloques, para no saturar la RAM
        auc.update_state(y[inicio:inicio + 2_000_000], p[inicio:inicio + 2_000_000])

    marcado = p > umbral
    verdaderos_positivos = float(np.sum(marcado & (y == 1)))
    falsos_positivos = float(np.sum(marcado & (y == 0)))
    falsos_negativos = float(np.sum(~marcado & (y == 1)))
    precision = verdaderos_positivos / max(verdaderos_positivos + falsos_positivos, 1.0)
    recall = verdaderos_positivos / max(verdaderos_positivos + falsos_negativos, 1.0)

    return {
        "auc_pr": 100.0 * float(auc.result()),
        "precision": 100.0 * precision,
        "recall": 100.0 * recall,
        "pixeles_validos": int(len(y)),
        "pixeles_con_fuego": int(np.sum(y == 1)),
    }


def reducir_resolucion(mascara, factor):
    """Pasa una máscara de 1 km a una de `factor` km (etiqueta "binaria").

    Regla del artículo (Sección V-A, Tabla III): un bloque de factor x factor
    píxeles es "fuego" si AL MENOS UNO de sus píxeles lo es. Si todos los
    píxeles del bloque son inciertos, el bloque queda incierto (-1).

    Args:
        mascara: (N, alto, ancho, 1) con valores -1, 0 o 1.

    Returns:
        (N, alto / factor, ancho / factor, 1) con valores -1, 0 o 1.
    """
    n, alto, ancho, _ = mascara.shape
    bloques = mascara.reshape(n, alto // factor, factor, ancho // factor, factor)
    hay_fuego = (bloques == 1).any(axis=(2, 4))
    hay_dato = (bloques != -1).any(axis=(2, 4))
    salida = np.where(hay_fuego, 1.0, np.where(hay_dato, 0.0, -1.0))
    return salida[..., np.newaxis].astype(np.float32)


### Entrenamiento y evaluación (`src/entrenamiento.py`)

In [ ]:
"""
entrenamiento.py — Entrenar y evaluar la red neuronal.

Reproduce la Sección IV-B del artículo ("Training Details"):
    - Optimizador Adam con tasa de aprendizaje 0,0001.
    - Pérdida: entropía cruzada ponderada (peso 3 para "fuego").
    - Aumento de datos: recorte, volteo y rotación al azar.
    - Se conserva el modelo con mejor AUC PR en el conjunto de validación.

OJO: los autores publicaron el modelo, la pérdida y las métricas, pero NO el
script de entrenamiento. Este archivo lo reconstruye a partir del texto.
"""

import csv
import json
import os
import time

import keras
import numpy as np


def _canales(variables):
    """Posición de cada variable pedida dentro de los 12 canales."""
    return [VARIABLES_ENTRADA.index(v) for v in variables]


def sigmoide(logits):
    """Convierte logits en probabilidades entre 0 y 1."""
    return 1.0 / (1.0 + np.exp(-logits))


def guardar_json(datos, nombre_archivo):
    """Guarda un diccionario en resultados/<nombre_archivo>."""
    os.makedirs(DIR_RESULTADOS, exist_ok=True)
    ruta = os.path.join(DIR_RESULTADOS, nombre_archivo)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(datos, f, indent=2, ensure_ascii=False)
    return ruta


def entrenar_red(variables=VARIABLES_ENTRADA, nombre="red_todas", epocas=None,
                 pasos_por_epoca=None, tasa_aprendizaje=TASA_APRENDIZAJE, verbose=2):
    """Entrena el autoencoder con las variables indicadas.

    Args:
        variables: qué canales de entrada usar (los 12, o menos en la ablación).
        nombre: prefijo de los archivos que se guardan.
        epocas, pasos_por_epoca: si no se dan, salen del presupuesto elegido en
            config.py.
        tasa_aprendizaje: por defecto, la del artículo (0,0001).

    Returns:
        (modelo con los pesos de su MEJOR época, historial de métricas).
    """
    presupuesto = PRESUPUESTOS[PRESUPUESTO]
    epocas = epocas or presupuesto["epocas"]
    pasos_por_epoca = pasos_por_epoca or presupuesto["pasos_por_epoca"]

    # Misma semilla -> mismos pesos iniciales y mismo orden de los datos.
    keras.utils.set_random_seed(SEMILLA)

    modelo = crear_modelo(num_canales=len(variables))
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=tasa_aprendizaje),
        loss=perdida_ponderada_con_mascara(),
        metrics=metricas_de_entrenamiento(),
    )

    # Validación: un recorte fijo de 32 x 32 por cada muestra de 'eval'.
    X_val, Y_val = cargar_numpy("eval")
    X_val, Y_val = recortes_fijos(X_val[..., _canales(variables)], Y_val)

    # Tres ayudantes que Keras llama al final de cada época:
    os.makedirs(DIR_MODELOS, exist_ok=True)
    ruta_pesos = os.path.join(DIR_MODELOS, f"{nombre}.weights.h5")
    ruta_registro = os.path.join(DIR_MODELOS, f"{nombre}_epocas.csv")
    mejor_hasta_ahora = max(
        (fila["val_auc_pr"] for fila in _leer_registro(ruta_registro)), default=None
    )
    ayudantes = [
        # 1. Si la sesión se corta (pasa en Colab y Kaggle), al volver a llamar
        #    a esta función el entrenamiento sigue desde la última época
        #    terminada en vez de empezar de cero.
        keras.callbacks.BackupAndRestore(os.path.join(DIR_MODELOS, f"{nombre}_respaldo")),
        # 2. Igual que el BestModelExporter de los autores: solo se guardan los
        #    pesos cuando mejora el AUC PR de validación.
        keras.callbacks.ModelCheckpoint(
            ruta_pesos, monitor="val_auc_pr", mode="max", save_best_only=True,
            save_weights_only=True, initial_value_threshold=mejor_hasta_ahora,
        ),
        # 3. Anota las métricas de cada época en un CSV (sobrevive a un corte).
        keras.callbacks.CSVLogger(ruta_registro, append=True),
    ]

    inicio = time.time()
    modelo.fit(
        dataset_entrenamiento(variables),
        steps_per_epoch=pasos_por_epoca,
        epochs=epocas,
        validation_data=(X_val, Y_val),
        callbacks=ayudantes,
        verbose=verbose,
    )
    minutos = (time.time() - inicio) / 60.0

    # El historial completo sale del CSV (incluye las épocas previas a un corte).
    filas = _leer_registro(ruta_registro)
    historial = {clave: [fila[clave] for fila in filas] for clave in filas[0] if clave != "epoch"}

    # Se recupera la mejor época (no necesariamente la última).
    modelo.load_weights(ruta_pesos)
    mejor_epoca = int(np.argmax(historial["val_auc_pr"])) + 1

    guardar_json(
        {
            "nombre": nombre,
            "variables": list(variables),
            "epocas": epocas,
            "pasos_por_epoca": pasos_por_epoca,
            "pasos_totales": epocas * pasos_por_epoca,
            "tasa_aprendizaje": tasa_aprendizaje,
            "minutos_ultimo_tramo": round(minutos, 1),
            "mejor_epoca": mejor_epoca,
            "mejor_val_auc_pr": 100.0 * float(max(historial["val_auc_pr"])),
            "parametros": int(modelo.count_params()),
            "historial": historial,
        },
        f"historial_{nombre}.json",
    )
    return modelo, historial


def _leer_registro(ruta):
    """Lee el CSV de épocas como lista de diccionarios de números."""
    if not os.path.exists(ruta):
        return []
    with open(ruta, newline="") as f:
        return [{k: float(v) for k, v in fila.items()} for fila in csv.DictReader(f)]


def cargar_red(variables=VARIABLES_ENTRADA, nombre="red_todas"):
    """Reconstruye una red ya entrenada a partir de sus pesos guardados."""
    modelo = crear_modelo(num_canales=len(variables))
    modelo.load_weights(os.path.join(DIR_MODELOS, f"{nombre}.weights.h5"))
    return modelo


def predecir_red(modelo, X, variables=VARIABLES_ENTRADA):
    """Probabilidad de fuego por píxel para un arreglo de muestras."""
    logits = modelo.predict(X[..., _canales(variables)], batch_size=256, verbose=0)
    return sigmoide(logits)


def evaluar_red(modelo, X, Y, variables=VARIABLES_ENTRADA,
                repeticiones=REPETICIONES_PRUEBA):
    """Mide la red sobre un conjunto (normalmente el de prueba).

    Se reportan dos protocolos:
      - "completo_64": la muestra entera de 64 x 64 km. Es el que se compara
        con la Tabla II: con él, la persistencia reproduce exactamente los
        números del artículo (paso 3). La red puede recibir 64 x 64 aunque se
        entrenó con 32 x 32 porque solo tiene capas convolucionales.
      - "recortes_32": recortes aleatorios fijos de 32 x 32 km, el tamaño con
        el que se entrena. Sirve para ver cuánto cambia el número según los
        píxeles sobre los que se mide.
    """
    X_rec, Y_rec = recortes_fijos(X, Y, repeticiones=repeticiones)
    prob_rec = predecir_red(modelo, X_rec, variables)
    en_recortes = calcular_metricas(Y_rec, prob_rec)

    # ¿Cuánto cambia el AUC PR solo por el azar del recorte? (el artículo
    # reporta una variación de 0,5 puntos entre experimentos, Sección V-B).
    n = len(X)
    aucs = [
        calcular_metricas(Y_rec[k * n:(k + 1) * n], prob_rec[k * n:(k + 1) * n])["auc_pr"]
        for k in range(repeticiones)
    ]
    en_recortes["auc_pr_por_repeticion"] = aucs
    en_recortes["auc_pr_desviacion"] = float(np.std(aucs))

    en_completo = calcular_metricas(Y, predecir_red(modelo, X, variables))
    return {"recortes_32": en_recortes, "completo_64": en_completo}


### Modelos base (`src/modelos_base.py`)
Persistencia, regresión logística y random forest.

In [ ]:
"""
modelos_base.py — Modelos de comparación: persistencia, regresión logística y
random forest.

Reproduce la Sección IV-A y el Apéndice A-B del artículo.

A diferencia de la red, estos modelos NO ven la imagen completa: clasifican
píxel por píxel. Para darles algo de contexto espacial, cada píxel se describe
con sus 12 variables y las de sus 8 vecinos (un cuadrado de 3 x 3):

        1 2 3
        4 5 6      9 posiciones x 12 variables = 108 columnas
        7 8 9      (la posición 5 es el propio píxel)

Detalles tomados del artículo:
  - Entrenamiento balanceado ~1:1 entre píxeles con y sin fuego.
  - Se excluye el borde de 1 píxel (no tiene los 8 vecinos).
  - Los píxeles inciertos (-1) no se usan.
  - La clase "sin fuego" pesa (N / N_sin_fuego) x W, con W = 7 para la
    regresión logística y W = 5 para el random forest.
  - Random forest con profundidad máxima 15; lo demás, valores por defecto
    de Scikit-learn.
"""

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


NUM_VARIABLES = len(VARIABLES_ENTRADA)
_INDICE_FUEGO_PREVIO = VARIABLES_ENTRADA.index("PrevFireMask")

# Desplazamientos (fila, columna) de las 9 posiciones, en el orden 1..9.
_VECINOS = [(df, dc) for df in (-1, 0, 1) for dc in (-1, 0, 1)]


def nombres_columnas():
    """Lista de 108 pares (variable, posición 1..9), en el orden de las columnas."""
    return [(v, pos + 1) for pos in range(9) for v in VARIABLES_ENTRADA]


def columnas_de(variable):
    """Índices de las 9 columnas (una por posición) de una variable."""
    c = VARIABLES_ENTRADA.index(variable)
    return [pos * NUM_VARIABLES + c for pos in range(9)]


# ---------------------------------------------------------------------------
# 1. PERSISTENCIA: "mañana habrá fuego donde hoy hay fuego"
# ---------------------------------------------------------------------------


def predecir_persistencia(X):
    """Devuelve la máscara de fuego de hoy como predicción para mañana.

    No aprende nada: es el piso que cualquier modelo debe superar. Los píxeles
    inciertos de hoy (-1) se cuentan como "sin fuego".
    """
    return np.clip(X[..., _INDICE_FUEGO_PREVIO:_INDICE_FUEGO_PREVIO + 1], 0.0, 1.0)


# ---------------------------------------------------------------------------
# 2. DE IMÁGENES A TABLAS DE PÍXELES
# ---------------------------------------------------------------------------


def vecindades_de_pixeles(X, n, fila, columna):
    """Columnas 3 x 3 de un conjunto de píxeles elegidos.

    Args:
        X: (N, alto, ancho, 12).
        n, fila, columna: índices de los píxeles (deben estar fuera del borde).

    Returns:
        (cantidad de píxeles, 108)
    """
    partes = [X[n, fila + df, columna + dc, :] for df, dc in _VECINOS]
    return np.concatenate(partes, axis=1)


def vecindades_completas(X):
    """Columnas 3 x 3 de TODOS los píxeles interiores de cada imagen.

    Returns:
        (N * (alto - 2) * (ancho - 2), 108)
    """
    _, alto, ancho, _ = X.shape
    partes = [
        X[:, 1 + df:alto - 1 + df, 1 + dc:ancho - 1 + dc, :] for df, dc in _VECINOS
    ]
    return np.concatenate(partes, axis=-1).reshape(-1, 9 * NUM_VARIABLES)


def muestrear_pixeles_entrenamiento(max_pixeles, semilla=SEMILLA):
    """Arma la tabla de entrenamiento balanceada 1:1 a partir de 'train'.

    Recorre el conjunto de entrenamiento por bloques. De cada bloque toma
    todos los píxeles con fuego y la misma cantidad de píxeles sin fuego
    elegidos al azar. Al final, si hay más de `max_pixeles`, se queda con una
    muestra aleatoria que mantiene el balance.

    Returns:
        caracteristicas: (M, 108) float32
        etiquetas: (M,) con 0 (sin fuego) o 1 (fuego)
    """
    azar = np.random.default_rng(semilla)
    con_fuego, sin_fuego = [], []
    for X, Y in iterar_numpy("train", tam_bloque=512):
        sin_borde = Y[:, 1:-1, 1:-1, 0]
        for valor, destino in ((1, con_fuego), (0, sin_fuego)):
            n, f, c = np.nonzero(sin_borde == valor)
            if valor == 0:  # igualar la cantidad de píxeles con fuego del bloque
                cuantos = min(int(np.sum(sin_borde == 1)), len(n))
                elegidos = azar.choice(len(n), size=cuantos, replace=False)
                n, f, c = n[elegidos], f[elegidos], c[elegidos]
            # +1 porque `sin_borde` empieza en la fila/columna 1 de la imagen.
            destino.append(vecindades_de_pixeles(X, n, f + 1, c + 1))

    con_fuego, sin_fuego = np.concatenate(con_fuego), np.concatenate(sin_fuego)
    por_clase = min(len(con_fuego), len(sin_fuego), max_pixeles // 2)
    con_fuego = con_fuego[azar.choice(len(con_fuego), por_clase, replace=False)]
    sin_fuego = sin_fuego[azar.choice(len(sin_fuego), por_clase, replace=False)]

    caracteristicas = np.concatenate([con_fuego, sin_fuego])
    etiquetas = np.concatenate([np.ones(por_clase), np.zeros(por_clase)]).astype(np.int8)
    orden = azar.permutation(len(etiquetas))
    return caracteristicas[orden], etiquetas[orden]


# ---------------------------------------------------------------------------
# 3. ENTRENAR Y PREDECIR
# ---------------------------------------------------------------------------


def pesos_de_clase(etiquetas, w):
    """Peso de la clase "sin fuego" según el Apéndice A-B del artículo.

        peso_sin_fuego = (número de ejemplos / número de ejemplos sin fuego) x W

    Con datos balanceados 1:1 la fracción vale 2, así que el peso es 2 x W.
    Subir W hace al modelo más "prudente": marca menos fuego, con lo que sube
    la precisión y baja el recall.
    """
    fraccion = len(etiquetas) / max(int(np.sum(etiquetas == 0)), 1)
    return {0: fraccion * w, 1: 1.0}


def entrenar_logistica(caracteristicas, etiquetas, w):
    """Regresión logística con los hiperparámetros por defecto de Scikit-learn."""
    modelo = LogisticRegression(class_weight=pesos_de_clase(etiquetas, w))
    return modelo.fit(caracteristicas, etiquetas)


def entrenar_bosque(caracteristicas, etiquetas, w, nucleos=-1):
    """Random forest con profundidad máxima 15 (Apéndice A-B)."""
    modelo = RandomForestClassifier(
        max_depth=PROFUNDIDAD_BOSQUE,
        class_weight=pesos_de_clase(etiquetas, w),
        n_jobs=nucleos,
        random_state=SEMILLA,
    )
    return modelo.fit(caracteristicas, etiquetas)


def predecir_por_pixel(modelo, X, columnas=None, tam_bloque=256):
    """Probabilidad de fuego para cada píxel interior de cada imagen.

    Args:
        modelo: clasificador de Scikit-learn ya entrenado.
        X: (N, alto, ancho, 12).
        columnas: subconjunto de las 108 columnas (para la ablación).

    Returns:
        (N, alto - 2, ancho - 2, 1)
    """
    n, alto, ancho, _ = X.shape
    salidas = []
    for inicio in range(0, n, tam_bloque):  # por bloques, para no saturar la RAM
        tabla = vecindades_completas(X[inicio:inicio + tam_bloque])
        if columnas is not None:
            tabla = tabla[:, columnas]
        salidas.append(modelo.predict_proba(tabla)[:, 1])
    return np.concatenate(salidas).reshape(n, alto - 2, ancho - 2, 1).astype(np.float32)


def interior(Y):
    """Quita el borde de 1 píxel a las etiquetas, igual que a las predicciones."""
    return Y[:, 1:-1, 1:-1, :]


### Gráficos (`src/graficos.py`)

In [ ]:
"""
graficos.py — Figuras de la replicación (Matplotlib).

Todas las funciones guardan un PNG en resultados/ y devuelven su ruta.
Las máscaras de fuego usan la convención de colores del artículo:
rojo = fuego, gris = sin fuego, negro = dato incierto.
"""

import os

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.ticker import MaxNLocator


# Colores: una paleta de dos series (artículo / replicación) y tinta neutra.
COLOR_ARTICULO = "#2a78d6"     # azul
COLOR_REPLICA = "#eb6834"      # naranja
COLOR_FONDO = "#fcfcfb"
TINTA = "#0b0b0b"
TINTA_SUAVE = "#52514e"
REJILLA = "#e4e3df"

# Máscaras: -1 incierto (negro), 0 sin fuego (gris), 1 fuego (rojo).
MAPA_FUEGO = ListedColormap(["#1a1a19", "#c9c8c3", "#e8421c"])
NORMA_FUEGO = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], MAPA_FUEGO.N)

plt.rcParams.update({
    "figure.facecolor": COLOR_FONDO,
    "axes.facecolor": COLOR_FONDO,
    "savefig.facecolor": COLOR_FONDO,
    "font.size": 10,
    "text.color": TINTA,
    "axes.labelcolor": TINTA_SUAVE,
    "xtick.color": TINTA_SUAVE,
    "ytick.color": TINTA_SUAVE,
    "axes.edgecolor": REJILLA,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def _con_coma(valor):
    """Número con un decimal y coma decimal (28,4), como se escribe en español."""
    return f"{valor:.1f}".replace(".", ",")


def _guardar(figura, nombre):
    os.makedirs(DIR_RESULTADOS, exist_ok=True)
    ruta = os.path.join(DIR_RESULTADOS, nombre)
    figura.savefig(ruta, dpi=160, bbox_inches="tight")
    return ruta


def _mascara(eje, imagen, titulo=None):
    """Dibuja una máscara de fuego sin ejes."""
    eje.imshow(imagen, cmap=MAPA_FUEGO, norm=NORMA_FUEGO, interpolation="nearest")
    eje.set_xticks([])
    eje.set_yticks([])
    for borde in eje.spines.values():
        borde.set_visible(False)
    if titulo:
        eje.set_title(titulo, fontsize=9, color=TINTA)


def figura_ejemplos(X_crudo, Y, indices, nombre="fig1_ejemplos_dataset.png"):
    """Equivalente a la Fig. 1 del artículo: una fila por muestra.

    Columnas: las 11 variables (en sus unidades originales), el fuego de hoy y
    el fuego de mañana.
    """
    variables = [v for v in VARIABLES_ENTRADA if v != "PrevFireMask"]
    columnas = len(variables) + 2
    figura, ejes = plt.subplots(len(indices), columnas, figsize=(1.45 * columnas, 1.75 * len(indices)))
    for fila, i in enumerate(indices):
        for col, variable in enumerate(variables):
            eje = ejes[fila, col]
            imagen = eje.imshow(X_crudo[i, ..., VARIABLES_ENTRADA.index(variable)], cmap="viridis")
            eje.set_xticks([])
            eje.set_yticks([])
            barra = figura.colorbar(imagen, ax=eje, orientation="horizontal", pad=0.04, fraction=0.08)
            barra.ax.tick_params(labelsize=6)
            barra.ax.locator_params(nbins=2)
            if fila == 0:
                eje.set_title(NOMBRES_ES[variable].replace(" ", "\n", 1), fontsize=8, color=TINTA)
        hoy = X_crudo[i, ..., VARIABLES_ENTRADA.index("PrevFireMask")]
        _mascara(ejes[fila, -2], hoy, "Fuego hoy\n(día t)" if fila == 0 else None)
        _mascara(ejes[fila, -1], Y[i, ..., 0], "Fuego mañana\n(día t+1)" if fila == 0 else None)
    figura.suptitle("Muestras del dataset (64 x 64 km, 1 km por píxel). "
                    "Rojo: fuego · gris: sin fuego · negro: incierto", fontsize=10, y=1.0)
    figura.tight_layout()
    return _guardar(figura, nombre)


def figura_curvas(historial, mejor_epoca, nombre="fig2_curvas_entrenamiento.png"):
    """Pérdida y AUC PR por época, en entrenamiento y validación."""
    epocas = np.arange(1, len(historial["loss"]) + 1)
    figura, (eje1, eje2) = plt.subplots(1, 2, figsize=(10, 3.6))
    for eje, clave, titulo in ((eje1, "loss", "Pérdida (entropía cruzada ponderada)"),
                               (eje2, "auc_pr", "AUC PR (%)")):
        escala = 100.0 if clave == "auc_pr" else 1.0
        eje.plot(epocas, escala * np.array(historial[clave]), color=COLOR_ARTICULO, lw=2, label="Entrenamiento")
        eje.plot(epocas, escala * np.array(historial["val_" + clave]), color=COLOR_REPLICA, lw=2, label="Validación")
        eje.axvline(mejor_epoca, color=TINTA_SUAVE, lw=1, ls=":")
        eje.set_title(titulo, loc="left", fontsize=10)
        eje.set_xlabel("Época")
        eje.xaxis.set_major_locator(MaxNLocator(integer=True))
        eje.grid(axis="y", color=REJILLA, lw=0.8)
        eje.legend(frameon=False)
    eje2.annotate(f"mejor época: {mejor_epoca}", xy=(mejor_epoca, eje2.get_ylim()[0]),
                  xytext=(4, 6), textcoords="offset points", fontsize=8, color=TINTA_SUAVE)
    figura.tight_layout()
    return _guardar(figura, nombre)


def figura_predicciones(X, Y, probabilidad, indices, titulos=None, nombre="fig3_predicciones.png"):
    """Equivalente a la Fig. 3 del artículo: hoy, mañana real y mañana predicho.

    Args:
        indices: qué muestras dibujar (una por columna).
        titulos: texto opcional encima de cada columna.
    """
    figura, ejes = plt.subplots(3, len(indices), figsize=(1.5 * len(indices), 4.9))
    rotulos = ("Fuego hoy\n(entrada)", "Fuego mañana\n(real)", "Fuego mañana\n(predicho)")
    canal = VARIABLES_ENTRADA.index("PrevFireMask")
    for col, i in enumerate(indices):
        _mascara(ejes[0, col], X[i, ..., canal], titulos[col] if titulos else None)
        _mascara(ejes[1, col], Y[i, ..., 0])
        _mascara(ejes[2, col], (probabilidad[i, ..., 0] > 0.5).astype(float))
    for fila, rotulo in enumerate(rotulos):
        ejes[fila, 0].set_ylabel(rotulo, fontsize=9, color=TINTA, rotation=0, ha="right", va="center")
    figura.suptitle("Predicciones de la red en el conjunto de prueba (recortes de 32 x 32 km)", fontsize=10)
    figura.tight_layout()
    return _guardar(figura, nombre)


def figura_comparacion(filas, nombre="fig4_comparacion_tabla2.png"):
    """Barras agrupadas: artículo vs. replicación para cada modelo y métrica.

    Args:
        filas: lista de (nombre del modelo, (auc, prec, rec) artículo,
            (auc, prec, rec) replicación). El nombre puede traer saltos de línea.
    """
    metricas = ("AUC PR (%)", "Precisión (%)", "Recall (%)")
    figura, ejes = plt.subplots(1, 3, figsize=(14, 4.0), sharey=False)
    posiciones = np.arange(len(filas))
    ancho = 0.36
    for k, (eje, titulo) in enumerate(zip(ejes, metricas)):
        del_articulo = [f[1][k] for f in filas]
        propios = [f[2][k] for f in filas]
        b1 = eje.bar(posiciones - ancho / 2 - 0.01, del_articulo, ancho, color=COLOR_ARTICULO, label="Artículo")
        b2 = eje.bar(posiciones + ancho / 2 + 0.01, propios, ancho, color=COLOR_REPLICA, label="Replicación")
        eje.bar_label(b1, fmt=_con_coma, fontsize=8, color=TINTA, padding=2)
        eje.bar_label(b2, fmt=_con_coma, fontsize=8, color=TINTA, padding=2)
        eje.set_xticks(posiciones)
        eje.set_xticklabels([f[0] for f in filas], fontsize=8, color=TINTA)
        eje.set_title(titulo, loc="left", fontsize=10)
        eje.grid(axis="y", color=REJILLA, lw=0.8)
        eje.set_axisbelow(True)
        eje.set_ylim(0, max(del_articulo + propios) * 1.18)
    ejes[0].legend(frameon=False, loc="upper right")
    figura.suptitle("Tabla II del artículo frente a esta replicación (conjunto de prueba)", fontsize=11, x=0.01, ha="left")
    figura.tight_layout()
    return _guardar(figura, nombre)


def figura_importancia(del_articulo, propias, nombre="fig5_importancia_variables.png"):
    """Importancia de Gini del random forest sumada por variable.

    Args:
        del_articulo: dict variable -> importancia sumada en las 9 posiciones.
            Solo trae las variables que el artículo publica completas.
        propias: dict variable -> importancia sumada en las 9 posiciones.
    """
    orden = sorted(propias, key=lambda v: propias[v])
    y = np.arange(len(orden))
    alto = 0.36
    figura, eje = plt.subplots(figsize=(8, 5))
    eje.barh(y + alto / 2 + 0.01, [del_articulo.get(v, 0.0) for v in orden], alto,
             color=COLOR_ARTICULO, label="Artículo (solo publica completas estas dos variables)")
    eje.barh(y - alto / 2 - 0.01, [propias[v] for v in orden], alto,
             color=COLOR_REPLICA, label="Replicación")
    eje.set_yticks(y)
    eje.set_yticklabels([NOMBRES_ES[v] for v in orden], color=TINTA)
    eje.set_xlabel("Importancia de Gini (suma de las 9 posiciones de la vecindad)")
    eje.set_title("Random forest: ¿qué variables pesan más?", loc="left", fontsize=11)
    eje.grid(axis="x", color=REJILLA, lw=0.8)
    eje.set_axisbelow(True)
    eje.legend(frameon=False, loc="lower right")
    figura.tight_layout()
    return _guardar(figura, nombre)


## Paso 1 — Explorar el dataset (Sección II del artículo)

Antes de entrenar nada se comprueba que los datos descargados de Kaggle son
los que describe el artículo:

| Dato | Artículo |
| --- | --- |
| Muestras | 18 545 |
| El fuego crece de t a t+1 | 58 % (10 798 muestras) |
| El fuego decrece | 39 % (7 191 muestras) |
| Tamaño de cada muestra | 64 x 64 km, 12 capas de entrada + 1 de salida |

In [ ]:
import os
import sys


import numpy as np


CANAL_FUEGO = VARIABLES_ENTRADA.index("PrevFireMask")


### 1.1 Recorrer las tres particiones
Se leen los datos SIN normalizar y se acumulan conteos y estadísticas.

In [ ]:
resumen = {}
conteo_etiquetas = {-1: 0, 0: 0, 1: 0}
crece = decrece = igual = centro_con_fuego = sin_fuego_previo = 0
suma = np.zeros(len(VARIABLES_ENTRADA))
suma_cuadrados = np.zeros(len(VARIABLES_ENTRADA))
pixeles_train = 0
minimos = np.full(len(VARIABLES_ENTRADA), np.inf)
maximos = np.full(len(VARIABLES_ENTRADA), -np.inf)
nan_total = 0

for particion in ("train", "eval", "test"):
    muestras = 0
    for x, y in leer_crudo(particion).batch(512):
        x, y = x.numpy(), y.numpy()
        muestras += len(x)
        nan_total += int(np.isnan(x).sum() + np.isnan(y).sum())

        # ¿Cuántos píxeles con fuego hay hoy y mañana en cada muestra?
        fuego_hoy = (x[..., CANAL_FUEGO] == 1).sum(axis=(1, 2))
        fuego_manana = (y[..., 0] == 1).sum(axis=(1, 2))
        crece += int((fuego_manana > fuego_hoy).sum())
        decrece += int((fuego_manana < fuego_hoy).sum())
        igual += int((fuego_manana == fuego_hoy).sum())
        sin_fuego_previo += int((fuego_hoy == 0).sum())

        # ¿El píxel central de la muestra tiene fuego mañana? (ver nota abajo)
        centro_con_fuego += int((y[:, 32, 32, 0] == 1).sum())

        for valor in conteo_etiquetas:
            conteo_etiquetas[valor] += int((y == valor).sum())

        minimos = np.minimum(minimos, x.min(axis=(0, 1, 2)))
        maximos = np.maximum(maximos, x.max(axis=(0, 1, 2)))

        if particion == "train":
            # Media y desviación DESPUÉS de recortar, como hicieron los autores.
            limites = np.array([ESTADISTICAS[v][:2] for v in VARIABLES_ENTRADA])
            recortado = np.clip(x, limites[:, 0], limites[:, 1]).astype(np.float64)
            suma += recortado.sum(axis=(0, 1, 2))
            suma_cuadrados += (recortado ** 2).sum(axis=(0, 1, 2))
            pixeles_train += recortado.shape[0] * recortado.shape[1] * recortado.shape[2]
    resumen[particion] = muestras

total = sum(resumen.values())
media = suma / pixeles_train
desviacion = np.sqrt(suma_cuadrados / pixeles_train - media ** 2)


### 1.2 Comparar con lo que dice el artículo

In [ ]:
print(f"Muestras: train {resumen['train']:,} | eval {resumen['eval']:,} | test {resumen['test']:,} "
      f"| total {total:,} (artículo: {ARTICULO_DATASET['muestras']:,})")
print(f"Reparto: {resumen['train'] / total:.1%} / {resumen['eval'] / total:.1%} / {resumen['test'] / total:.1%} (artículo: 8:1:1)")
print(f"El fuego crece:   {crece:,} ({crece / total:.1%})   artículo: {ARTICULO_DATASET['fuego_crece']:,} (58 %)")
print(f"El fuego decrece: {decrece:,} ({decrece / total:.1%})   artículo: {ARTICULO_DATASET['fuego_decrece']:,} (39 %)")
print(f"Queda igual:      {igual:,} ({igual / total:.1%})")
print(f"Muestras sin ningún píxel de fuego hoy: {sin_fuego_previo:,}")
print(f"Valores NaN en todo el dataset: {nan_total:,}")

pixeles = sum(conteo_etiquetas.values())
print("\nEtiquetas (fuego de mañana), sobre todos los píxeles:")
for valor, texto in ((1, "fuego"), (0, "sin fuego"), (-1, "incierto")):
    print(f"  {texto:10s} {conteo_etiquetas[valor] / pixeles:7.2%}")
print(f"\nMuestras cuyo píxel central (32, 32) tiene fuego mañana: {centro_con_fuego / total:.1%}")

print("\nMedia y desviación en 'train' después de recortar (calculado aquí | código de los autores):")
for i, variable in enumerate(VARIABLES_ENTRADA[:-1]):
    _, _, media_autores, desv_autores = ESTADISTICAS[variable]
    print(f"  {NOMBRES_ES[variable]:22s} media {media[i]:10.4f} | {media_autores:<10} desv {desviacion[i]:10.4f} | {desv_autores}")

guardar_json(
    {
        "muestras": resumen,
        "total": total,
        "fuego_crece": crece,
        "fuego_decrece": decrece,
        "fuego_igual": igual,
        "muestras_sin_fuego_previo": sin_fuego_previo,
        "nan": nan_total,
        "fraccion_etiquetas": {str(k): v / pixeles for k, v in conteo_etiquetas.items()},
        "fraccion_centro_con_fuego": centro_con_fuego / total,
        "estadisticas_train": {
            v: {"min": float(minimos[i]), "max": float(maximos[i]),
                "media_recortada": float(media[i]), "desv_recortada": float(desviacion[i])}
            for i, v in enumerate(VARIABLES_ENTRADA)
        },
    },
    "paso1_exploracion.json",
)


### 1.3 Ver algunas muestras (equivalente a la Fig. 1 del artículo)

In [ ]:
x, y = next(iter(leer_crudo("test").batch(64)))
figura_ejemplos(x.numpy(), y.numpy(), indices=[3, 12, 27, 41])


## Paso 2 — Entrenar la red neuronal (Secciones IV-A y IV-B del artículo)

Se entrena el autoencoder convolucional con las 12 variables de entrada y se
mide en el conjunto de prueba. El resultado se compara con la primera fila de
la Tabla II del artículo (AUC PR 28,4 %, precisión 33,6 %, recall 43,1 %).

In [ ]:
import os
import sys


NOMBRE = "red_todas"
terminado = os.path.join(DIR_RESULTADOS, f"historial_{NOMBRE}.json")

# Si el entrenamiento ya terminó se reutiliza el modelo (entrenar tarda). Para
# forzar uno nuevo: borrar resultados/modelos/ y resultados/historial_*.json.
# Si quedó a medias (se cortó la sesión), entrenar_red lo retoma donde quedó.
if os.path.exists(terminado):
    print("Se reutiliza el modelo ya entrenado:", NOMBRE)
    modelo = cargar_red(VARIABLES_ENTRADA, NOMBRE)
else:
    modelo, historial = entrenar_red(VARIABLES_ENTRADA, NOMBRE)


### Evaluación en el conjunto de prueba
Los 1 689 ejemplos de prueba nunca se usaron para entrenar ni para elegir el
modelo. Se mide sobre la muestra completa de 64 x 64 km, que es el protocolo
con el que la persistencia reproduce exactamente al artículo (paso 3).

In [ ]:
X_prueba, Y_prueba = cargar_numpy("test")
metricas = evaluar_red(modelo, X_prueba, Y_prueba)
guardar_json(metricas, f"metricas_{NOMBRE}.json")

m = metricas["completo_64"]
a = ARTICULO_TABLA_II["Red neuronal"]
print(f"{'':30s}{'AUC PR':>8s}{'Precisión':>11s}{'Recall':>8s}")
print(f"{'Artículo (Tabla II)':30s}{a[0]:8.1f}{a[1]:11.1f}{a[2]:8.1f}")
print(f"{'Esta replicación (64 x 64)':30s}{m['auc_pr']:8.1f}{m['precision']:11.1f}{m['recall']:8.1f}")
r = metricas["recortes_32"]
print(f"{'(sobre recortes de 32 x 32)':30s}{r['auc_pr']:8.1f}{r['precision']:11.1f}{r['recall']:8.1f}")
print(f"Variación del AUC PR entre recortes distintos: ±{r['auc_pr_desviacion']:.2f} puntos")


## Paso 3 — Modelos base (Sección IV-A y Apéndices A-B, A-C y A-D)

Se entrenan los tres modelos con los que el artículo compara a la red:

| Modelo | Qué hace | Artículo, Tabla II (AUC PR / precisión / recall) |
| --- | --- | --- |
| Persistencia | "mañana = hoy" | 11,5 / 35,7 / 27,3 |
| Regresión logística | píxel a píxel con vecindad 3 x 3 | 19,8 / 32,5 / 35,3 |
| Random forest | píxel a píxel con vecindad 3 x 3 | 22,5 / 26,3 / 46,9 |

In [ ]:
import os
import sys
import time


import warnings

import numpy as np
from sklearn.exceptions import ConvergenceWarning


# La regresión logística usa los valores por defecto de Scikit-learn (como en el
# artículo), que incluyen un máximo de 100 iteraciones. Algunos ajustes llegan a
# ese tope y Scikit-learn avisa; se silencia el aviso para no llenar la salida.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Conjunto de prueba: las 1 689 muestras completas de 64 x 64 km.
X_prueba, Y_prueba = cargar_numpy("test")

# Conjunto de validación, para elegir el peso W como hace el artículo.
X_val, Y_val = cargar_numpy("eval")


### 3.1 Persistencia (y verificación del protocolo de evaluación)
La persistencia no se entrena, así que su resultado solo depende de los datos
y de cómo se mide. Si coincide con el artículo, el protocolo de evaluación
está bien replicado. Se mide sobre la muestra completa de 64 x 64 km y
también a resolución más gruesa (filas "Persistence" de la Tabla III).

In [ ]:
CANAL_FUEGO = VARIABLES_ENTRADA.index("PrevFireMask")
persistencia = {"1x": calcular_metricas(Y_prueba, predecir_persistencia(X_prueba))}
for factor in (2, 4, 8):
    hoy = reducir_resolucion(X_prueba[..., CANAL_FUEGO, None], factor)
    manana = reducir_resolucion(Y_prueba, factor)
    persistencia[f"{factor}x"] = calcular_metricas(manana, np.clip(hoy, 0, 1))

# Mismo cálculo sobre recortes de 32 x 32 km: el número cambia solo por medir
# sobre otros píxeles (el fuego suele estar hacia el centro de la muestra).
X_rec, Y_rec = recortes_fijos(X_prueba, Y_prueba, repeticiones=3)
persistencia["recortes_32"] = calcular_metricas(Y_rec, predecir_persistencia(X_rec))
del X_rec, Y_rec

guardar_json(persistencia, "metricas_persistencia.json")
for clave, m in persistencia.items():
    print(f"Persistencia {clave:12s} AUC PR {m['auc_pr']:5.1f} | precisión {m['precision']:5.1f} | recall {m['recall']:5.1f}")


### 3.2 Tabla de píxeles para entrenar
Las imágenes se convierten en una tabla: una fila por píxel, 108 columnas.

In [ ]:
inicio = time.time()
tabla, etiquetas = muestrear_pixeles_entrenamiento(MAX_PIXELES_LOGISTICA)
print(f"Tabla de entrenamiento: {tabla.shape[0]:,} píxeles x {tabla.shape[1]} columnas "
      f"({etiquetas.mean():.0%} con fuego) en {time.time() - inicio:.0f} s")


### 3.3 Regresión logística
El artículo prueba W de 1 a 8 y se queda con el mejor AUC PR de validación
(obtuvo W = 7). Aquí se repite esa búsqueda.

In [ ]:
busqueda_w = {}
for w in range(1, 9):
    candidato = entrenar_logistica(tabla, etiquetas, w)
    busqueda_w[w] = calcular_metricas(interior(Y_val), predecir_por_pixel(candidato, X_val))
    m = busqueda_w[w]
    print(f"W = {w}: validación AUC PR {m['auc_pr']:.2f} | precisión {m['precision']:.1f} | recall {m['recall']:.1f}")
mejor_w = max(busqueda_w, key=lambda w: busqueda_w[w]["auc_pr"])
print(f"Mejor W por AUC PR de validación: {mejor_w} (el artículo eligió {W_LOGISTICA})")

# Para la tabla comparativa se usa el W del artículo (7), de modo que el punto
# de operación (precisión/recall) sea comparable con su Tabla II.
logistica = entrenar_logistica(tabla, etiquetas, W_LOGISTICA)
metricas_logistica = calcular_metricas(interior(Y_prueba), predecir_por_pixel(logistica, X_prueba))
# Como referencia, también con el W que salió mejor en nuestra validación.
metricas_mejor_w = calcular_metricas(
    interior(Y_prueba),
    predecir_por_pixel(entrenar_logistica(tabla, etiquetas, mejor_w), X_prueba),
)
guardar_json(
    {
        "prueba": metricas_logistica,
        "w_usado": W_LOGISTICA,
        "mejor_w_validacion": mejor_w,
        "prueba_con_mejor_w": metricas_mejor_w,
        "busqueda_w_validacion": busqueda_w,
        "pixeles_entrenamiento": int(len(etiquetas)),
        "iteraciones": int(logistica.n_iter_[0]),
    },
    "metricas_logistica.json",
)
print("Regresión logística (prueba):", {k: round(v, 1) for k, v in metricas_logistica.items() if k in ("auc_pr", "precision", "recall")})


### 3.4 Coeficientes de la regresión logística (Tabla VII del artículo)
Como las variables están normalizadas, el tamaño del coeficiente indica
cuánto pesa cada columna. El signo indica si empuja hacia "fuego" (+) o no (-).

In [ ]:
columnas = nombres_columnas()
orden = np.argsort(-np.abs(logistica.coef_[0]))[:20]
coeficientes = [
    {"variable": columnas[i][0], "posicion": columnas[i][1], "coeficiente": float(logistica.coef_[0][i])}
    for i in orden
]
guardar_json(coeficientes, "coeficientes_logistica.json")
for c in coeficientes[:10]:
    print(f"{c['variable']:14s} ({c['posicion']})  {c['coeficiente']:+.2f}")


### 3.5 Ablación de la regresión logística (Tabla VIII del artículo)
Se vuelve a entrenar 12 veces, quitando cada vez una variable (sus 9 columnas).

In [ ]:
ablacion_logistica = {}
todas = np.arange(tabla.shape[1])
for variable in VARIABLES_ENTRADA:
    quedan = np.setdiff1d(todas, columnas_de(variable))
    sin_variable = entrenar_logistica(tabla[:, quedan], etiquetas, W_LOGISTICA)
    prob = predecir_por_pixel(sin_variable, X_prueba, columnas=quedan)
    ablacion_logistica[variable] = calcular_metricas(interior(Y_prueba), prob)
    m = ablacion_logistica[variable]
    print(f"Sin {variable:14s} AUC PR {m['auc_pr']:5.1f} | precisión {m['precision']:5.1f} | recall {m['recall']:5.1f}")
guardar_json(ablacion_logistica, "ablacion_logistica.json")


### 3.6 Random forest
Se entrena con una muestra más pequeña de la tabla (ver `MAX_PIXELES_BOSQUE`)
porque es el modelo más lento en CPU. Se usa el W del artículo (5).

In [ ]:
azar = np.random.default_rng(0)
# Muestra balanceada: mitad de filas con fuego y mitad sin fuego.
filas = np.concatenate([
    azar.choice(np.nonzero(etiquetas == clase)[0], MAX_PIXELES_BOSQUE // 2, replace=False)
    for clase in (0, 1)
])
inicio = time.time()
bosque = entrenar_bosque(tabla[filas], etiquetas[filas], W_BOSQUE)
print(f"Random forest entrenado con {len(filas):,} píxeles en {(time.time() - inicio) / 60:.1f} min")

metricas_bosque = calcular_metricas(interior(Y_prueba), predecir_por_pixel(bosque, X_prueba))
guardar_json(
    {"prueba": metricas_bosque, "w_usado": W_BOSQUE, "pixeles_entrenamiento": int(len(filas))},
    "metricas_bosque.json",
)
print("Random forest (prueba):", {k: round(v, 1) for k, v in metricas_bosque.items() if k in ("auc_pr", "precision", "recall")})


### 3.7 Importancia de variables del random forest (Tabla VI del artículo)
La importancia de Gini mide cuánto ayuda cada columna a separar "fuego" de
"sin fuego" en los árboles. Todas las importancias suman 1.

In [ ]:
orden = np.argsort(-bosque.feature_importances_)
importancia = [
    {"variable": columnas[i][0], "posicion": columnas[i][1], "importancia": float(bosque.feature_importances_[i])}
    for i in orden
]
guardar_json(importancia, "importancia_bosque.json")
for item in importancia[:10]:
    print(f"{item['variable']:14s} ({item['posicion']})  {item['importancia']:.3f}")


## Paso 4 — ¿Qué variables necesita la red? (Sección V-B del artículo)

La red no dice por sí sola qué variables usa. El artículo lo averigua con
**ablación**: volver a entrenar quitando variables y ver cuánto empeora.

Con el presupuesto de este prototipo se replican los dos experimentos que
sostienen la conclusión principal del artículo:

| Experimento | Artículo (AUC PR / precisión / recall) |
| --- | --- |
| Quitar el fuego previo (Tabla IV, 1.ª fila) | 6,8 / 4,6 / 0,2 |
| Usar solo el fuego previo (texto de la Sección V-B) | 22,9 / 29,2 / 49,6 |

Las otras 11 filas de la Tabla IV y la Tabla V completa (22 entrenamientos
más) se pueden lanzar agregando entradas al diccionario `EXPERIMENTOS`.

In [ ]:
import os
import sys


EXPERIMENTOS = {
    # nombre del experimento -> variables de entrada que SÍ se usan
    "red_sin_fuego_previo": [v for v in VARIABLES_ENTRADA if v != "PrevFireMask"],
    "red_solo_fuego_previo": ["PrevFireMask"],
}

# Opcional: correr solo algunos, p. ej. WILDFIRE_EXPERIMENTOS=red_solo_fuego_previo
elegidos = os.environ.get("WILDFIRE_EXPERIMENTOS", ",".join(EXPERIMENTOS)).split(",")

X_prueba, Y_prueba = cargar_numpy("test")

for nombre in elegidos:
    variables = EXPERIMENTOS[nombre]
    if os.path.exists(os.path.join(DIR_RESULTADOS, f"historial_{nombre}.json")):
        print("Se reutiliza el modelo ya entrenado:", nombre)
        modelo = cargar_red(variables, nombre)
    else:
        # Mismo presupuesto y misma semilla que la red completa: así la única
        # diferencia entre los experimentos son las variables de entrada.
        modelo, _ = entrenar_red(variables, nombre)

    metricas = evaluar_red(modelo, X_prueba, Y_prueba, variables)
    guardar_json(metricas, f"metricas_{nombre}.json")
    m = metricas["completo_64"]
    print(f"{nombre}: AUC PR {m['auc_pr']:.1f} | precisión {m['precision']:.1f} | recall {m['recall']:.1f}")


## Paso 5 — Comparar con el artículo

Se juntan los resultados de los pasos anteriores y se ponen al lado de los
que publica el artículo. Salidas:

- `resultados/comparacion_resultados.md`: todas las tablas comparativas.
- `fig2` a `fig5`: curvas de entrenamiento, predicciones, comparación de la
  Tabla II e importancia de variables.

In [ ]:
import json
import os
import sys


import numpy as np


def leer(nombre):
    """Lee un JSON de la carpeta de resultados."""
    with open(os.path.join(DIR_RESULTADOS, nombre), encoding="utf-8") as f:
        return json.load(f)


def trio(m):
    """(AUC PR, precisión, recall) de un diccionario de métricas."""
    return (m["auc_pr"], m["precision"], m["recall"])


def num(valor, signo=False, decimales=1):
    """Número con coma decimal (28,4); con signo si es una diferencia."""
    valor = round(valor, decimales) + 0.0  # el + 0.0 evita que aparezca "-0,0"
    formato = f"+.{decimales}f" if signo and valor else f".{decimales}f"
    return format(valor, formato).replace(".", ",")


def miles(entero):
    """Entero con espacio como separador de miles (1 000 000)."""
    return format(entero, ",").replace(",", " ")


def fila(nombre, del_articulo, propio):
    """Una fila de tabla Markdown: artículo, replicación y diferencia."""
    celdas = [nombre]
    for a, p in zip(del_articulo, propio):
        celdas += [num(a), num(p), num(round(p, 1) - a, signo=True)]
    return "| " + " | ".join(celdas) + " |"


ENCABEZADO = (
    "| {} | AUC PR art. | AUC PR rép. | Δ | Prec. art. | Prec. rép. | Δ | Recall art. | Recall rép. | Δ |\n"
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |"
)


### 5.1 Tabla II — los cuatro modelos

In [ ]:
red = leer("metricas_red_todas.json")
historial = leer("historial_red_todas.json")
r_persistencia = leer("metricas_persistencia.json")
r_logistica = leer("metricas_logistica.json")
r_bosque = leer("metricas_bosque.json")

propios = {
    "Red neuronal": trio(red["completo_64"]),
    "Random forest": trio(r_bosque["prueba"]),
    "Regresión logística": trio(r_logistica["prueba"]),
    "Persistencia": trio(r_persistencia["1x"]),
}

lineas = ["# Comparación de resultados: artículo vs. replicación", ""]
lineas += [
    "Todas las métricas están en porcentaje y se midieron en el conjunto de prueba,",
    "sobre la muestra completa de 64 x 64 km (1 689 muestras).",
    "art. = artículo; rép. = esta replicación; Δ = replicación menos artículo (en puntos).",
    "",
    "## Tabla II — Predicción de la propagación", "",
    ENCABEZADO.format("Modelo"),
]
for modelo, del_articulo in ARTICULO_TABLA_II.items():
    lineas.append(fila(modelo, del_articulo, propios[modelo]))

# Las mismas predicciones medidas sobre recortes aleatorios de 32 x 32 km (el
# tamaño de entrenamiento). El artículo no dice sobre qué píxeles midió cada
# modelo, así que se muestran los dos protocolos.
r = red["recortes_32"]
lineas += ["", "Los mismos modelos medidos sobre recortes aleatorios de 32 x 32 km:", "",
           ENCABEZADO.format("Modelo (recortes de 32 x 32 km)"),
           fila("Red neuronal", ARTICULO_TABLA_II["Red neuronal"], trio(r)),
           fila("Persistencia", ARTICULO_TABLA_II["Persistencia"], trio(r_persistencia["recortes_32"]))]
lineas += [
    "",
    f"- Red neuronal: {miles(historial['pasos_totales'])} pasos de entrenamiento "
    f"({historial['epocas']} épocas x {historial['pasos_por_epoca']} pasos); "
    "el artículo usó 1 000 000 de pasos en 4 GPU V100.",
    f"- Mejor época según validación: {historial['mejor_epoca']} "
    f"(AUC PR de validación {num(historial['mejor_val_auc_pr'])} %).",
    f"- Variación del AUC PR de la red entre tres juegos de recortes distintos: "
    f"±{num(r['auc_pr_desviacion'], decimales=2)} puntos.",
    f"- Regresión logística: W = {r_logistica['w_usado']} (el del artículo), {miles(r_logistica['pixeles_entrenamiento'])} píxeles "
    f"de entrenamiento. El mejor W en nuestra validación fue {r_logistica['mejor_w_validacion']}; con él, en prueba: "
    f"AUC PR {num(r_logistica['prueba_con_mejor_w']['auc_pr'])} %, precisión {num(r_logistica['prueba_con_mejor_w']['precision'])} %, "
    f"recall {num(r_logistica['prueba_con_mejor_w']['recall'])} %.",
    f"- Random forest: W = {r_bosque['w_usado']}; entrenado con {miles(r_bosque['pixeles_entrenamiento'])} píxeles.",
]

figura_comparacion(
    [("Red neuronal\n(64 x 64)", ARTICULO_TABLA_II["Red neuronal"], propios["Red neuronal"]),
     ("Red neuronal\n(recortes 32 x 32)", ARTICULO_TABLA_II["Red neuronal"], trio(r))]
    + [(m.replace(" ", "\n", 1), ARTICULO_TABLA_II[m], propios[m]) for m in ARTICULO_TABLA_II if m != "Red neuronal"]
)


### 5.2 Tabla III — persistencia a resolución más gruesa

In [ ]:
lineas += ["", "## Tabla III (filas de persistencia) — Resolución más gruesa", "",
           ENCABEZADO.format("Resolución")]
for factor, del_articulo in ARTICULO_TABLA_III_PERSISTENCIA.items():
    lineas.append(fila(f"{factor}x ({factor} km)", del_articulo, trio(r_persistencia[f"{factor}x"])))


### 5.3 Ablación de la red (Tabla IV y Sección V-B)

In [ ]:
experimentos = (
    ("Las 12 variables (Tabla II)", "metricas_red_todas.json", ARTICULO_TABLA_II["Red neuronal"]),
    ("Sin el fuego previo (Tabla IV)", "metricas_red_sin_fuego_previo.json", ARTICULO_TABLA_IV["PrevFireMask"]),
    ("Solo el fuego previo (Sección V-B)", "metricas_red_solo_fuego_previo.json", ARTICULO_SOLO_FUEGO_PREVIO),
)
lineas += ["", "## Ablación de la red neuronal"]
for protocolo, titulo in (("completo_64", "muestra completa de 64 x 64 km"),
                          ("recortes_32", "recortes de 32 x 32 km")):
    lineas += ["", f"Medido sobre {titulo}:", "", ENCABEZADO.format("Variables de entrada")]
    for nombre, archivo, del_articulo in experimentos:
        if os.path.exists(os.path.join(DIR_RESULTADOS, archivo)):
            lineas.append(fila(nombre, del_articulo, trio(leer(archivo)[protocolo])))

# Experimento adicional (no está en el artículo): tasa de aprendizaje 10 veces mayor.
if os.path.exists(os.path.join(DIR_RESULTADOS, "metricas_red_todas_tasa_0001.json")):
    extra = leer("metricas_red_todas_tasa_0001.json")
    h_extra = leer("historial_red_todas_tasa_0001.json")
    lineas += ["", f"Experimento adicional, fuera del artículo: las 12 variables con tasa de aprendizaje 0,001 "
               f"y {h_extra['epocas']} épocas ({miles(h_extra['pasos_totales'])} pasos): "
               f"AUC PR {num(extra['completo_64']['auc_pr'])} / precisión {num(extra['completo_64']['precision'])} / "
               f"recall {num(extra['completo_64']['recall'])} sobre 64 x 64 km; "
               f"{num(extra['recortes_32']['auc_pr'])} / {num(extra['recortes_32']['precision'])} / "
               f"{num(extra['recortes_32']['recall'])} sobre recortes de 32 x 32 km."]


### 5.4 Ablación de la regresión logística (Tabla VIII)

In [ ]:
ablacion = leer("ablacion_logistica.json")
lineas += ["", "## Tabla VIII — Regresión logística quitando una variable", "",
           ENCABEZADO.format("Variable quitada")]
for variable, del_articulo in ARTICULO_TABLA_VIII.items():
    lineas.append(fila(NOMBRES_ES[variable], del_articulo, trio(ablacion[variable])))


### 5.5 Importancia de variables del random forest (Tabla VI)

In [ ]:
importancia = leer("importancia_bosque.json")
lineas += ["", "## Tabla VI — Random forest: 20 columnas más importantes (Gini)", "",
           "| # | Artículo | Importancia | Replicación | Importancia |",
           "| ---: | --- | ---: | --- | ---: |"]
for i, ((var_a, pos_a, imp_a), propia) in enumerate(zip(ARTICULO_TABLA_VI, importancia[:20]), start=1):
    lineas.append(f"| {i} | {NOMBRES_ES[var_a]} ({pos_a}) | {num(imp_a, decimales=3)} | "
                  f"{NOMBRES_ES[propia['variable']]} ({propia['posicion']}) | {num(propia['importancia'], decimales=3)} |")

# Suma por variable (las 9 posiciones juntas) para verlo en un solo gráfico.
suma_propia = {v: 0.0 for v in VARIABLES_ENTRADA}
for item in importancia:
    suma_propia[item["variable"]] += item["importancia"]
# Del artículo solo se puede sumar lo que publica: su Tabla VI trae las 20
# columnas más importantes. El fuego previo y el ERC aparecen con sus 9
# posiciones; la elevación solo con 2, así que no se incluye en el gráfico.
suma_articulo = {}
for variable in VARIABLES_ENTRADA:
    valores = [valor for v, _, valor in ARTICULO_TABLA_VI if v == variable]
    if len(valores) == 9:
        suma_articulo[variable] = sum(valores)

lineas += ["", "Suma de la importancia por variable (replicación, 9 posiciones):", "",
           "| Variable | Importancia |", "| --- | ---: |"]
for variable in sorted(suma_propia, key=lambda v: -suma_propia[v]):
    lineas.append(f"| {NOMBRES_ES[variable]} | {num(suma_propia[variable], decimales=3)} |")
figura_importancia(suma_articulo, suma_propia)


### 5.6 Coeficientes de la regresión logística (Tabla VII)

In [ ]:
coeficientes = leer("coeficientes_logistica.json")
lineas += ["", "## Tabla VII — Regresión logística: 20 coeficientes más grandes (replicación)", "",
           "| # | Columna | Coeficiente |", "| ---: | --- | ---: |"]
for i, c in enumerate(coeficientes, start=1):
    lineas.append(f"| {i} | {NOMBRES_ES[c['variable']]} ({c['posicion']}) | {num(c['coeficiente'], signo=True, decimales=2)} |")

ruta = os.path.join(DIR_RESULTADOS, "comparacion_resultados.md")
with open(ruta, "w", encoding="utf-8") as f:
    f.write("\n".join(lineas) + "\n")
print("\n".join(lineas))


### 5.7 Curvas de entrenamiento y ejemplos de predicción (Fig. 3 del artículo)

In [ ]:
figura_curvas(historial["historial"], historial["mejor_epoca"])

modelo = cargar_red(VARIABLES_ENTRADA, "red_todas")
X_prueba, Y_prueba = cargar_numpy("test")
X_rec, Y_rec = recortes_fijos(X_prueba, Y_prueba)
probabilidad = predecir_red(modelo, X_rec)

# Para elegir ejemplos se calcula el F1 de cada muestra (solo las que tienen
# al menos 20 km² de fuego real) y se muestran las 5 mejores y las 5 peores.
marcado = probabilidad[..., 0] > 0.5
real = Y_rec[..., 0] == 1
aciertos = (marcado & real).sum(axis=(1, 2))
f1 = 2 * aciertos / np.maximum(marcado.sum(axis=(1, 2)) + real.sum(axis=(1, 2)), 1)
candidatas = np.nonzero(real.sum(axis=(1, 2)) >= 20)[0]
orden = candidatas[np.argsort(-f1[candidatas])]
# (Se dibujan recortes de 32 x 32 km, como las figuras del artículo.)
figura_predicciones(X_rec, Y_rec, probabilidad, list(orden[:5]) + list(orden[-5:]),
                    titulos=["mejor"] * 5 + ["peor"] * 5)
print("Columnas 1-5: mejores predicciones (F1 más alto). Columnas 6-10: peores.")
